In [10]:
import asyncio
import aiohttp
import json
import logging
import re
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional


# ================================================================
#  CONFIG  ← edit these values before running
# ================================================================
INPUT_FILE    = "fujitsu_products.json"  # path to your JSON file
OUTPUT_DIR    = "fujitsu_products_html"            # folder to save .html files
CONCURRENCY   = 10                      # max parallel requests
TIMEOUT       = 30                      # seconds per request
RETRIES       = 3                       # attempts per URL
RETRY_DELAY   = 1.5                     # exponential back-off base (seconds)
SKIP_EXISTING = True                    # skip URLs already saved to disk
VERBOSE       = False                   # set True for debug logs
# ================================================================

folder_to_zip = '/content/fujitsu_products_html'
output_zip_file = '/content/fujitsu_products_html'

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
}


# ---------- LOGGING ----------
def setup_logging() -> logging.Logger:
    level = logging.DEBUG if VERBOSE else logging.INFO
    logging.basicConfig(
        level=level,
        format="%(asctime)s [%(levelname)s] %(message)s",
        datefmt="%H:%M:%S",
        force=True,  # re-apply in Colab which pre-configures logging
    )
    return logging.getLogger("scraper")


# ---------- DATA MODELS ----------
@dataclass
class Product:
    product_name: str
    product_url: str
    reference: str = ""
    brand: str = ""
    price: str = ""
    availability: str = ""

    @classmethod
    def from_dict(cls, data: dict) -> "Product":
        required = {"product_name", "product_url"}
        missing = required - data.keys()
        if missing:
            raise ValueError(f"Missing required fields: {missing}")
        return cls(**{k: data.get(k, "") for k in cls.__dataclass_fields__})


@dataclass
class Stats:
    total: int = 0
    success: int = 0
    failed: int = 0
    skipped: int = 0
    start_time: float = field(default_factory=time.time)

    def elapsed(self) -> float:
        return time.time() - self.start_time

    def summary(self) -> str:
        return (
            f"\n{'─'*45}\n"
            f"  Total    : {self.total}\n"
            f"  ✅ Saved  : {self.success}\n"
            f"  ⏭  Skipped: {self.skipped}\n"
            f"  ❌ Failed : {self.failed}\n"
            f"  Time     : {self.elapsed():.1f}s\n"
            f"{'─'*45}"
        )


# ---------- HELPERS ----------
def safe_filename(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    return text.strip("_")[:80]


def build_filename(product: Product) -> str:
    parts = [safe_filename(product.product_name)]
    if product.reference:
        parts.append(safe_filename(product.reference))
    return "_".join(parts) + ".html"


def load_products(json_path: str) -> list[Product]:
    path = Path(json_path)
    if not path.exists():
        raise FileNotFoundError(f"Input file not found: {json_path}")
    if path.suffix.lower() != ".json":
        raise ValueError(f"Expected a .json file, got: {json_path}")

    with open(path, encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, list):
        raise ValueError("JSON root must be a list of product objects.")

    products, errors = [], []
    for i, item in enumerate(data):
        try:
            products.append(Product.from_dict(item))
        except (ValueError, TypeError) as e:
            errors.append(f"  Item #{i}: {e}")

    if errors:
        logging.getLogger("scraper").warning(
            "Skipped %d invalid item(s):\n%s", len(errors), "\n".join(errors)
        )
    return products


# ---------- FETCH ----------
async def fetch_html(
    session: aiohttp.ClientSession,
    url: str,
    logger: logging.Logger,
) -> Optional[str]:
    for attempt in range(1, RETRIES + 1):
        try:
            async with session.get(url) as response:
                if response.status == 200:
                    return await response.text()
                if response.status in (403, 404, 410):
                    logger.warning("HTTP %s (no retry) -> %s", response.status, url)
                    return None
                logger.warning("HTTP %s (attempt %d/%d) -> %s", response.status, attempt, RETRIES, url)
        except asyncio.TimeoutError:
            logger.warning("Timeout (attempt %d/%d) -> %s", attempt, RETRIES, url)
        except aiohttp.ClientConnectorError as e:
            logger.warning("Connection error (attempt %d/%d) -> %s: %s", attempt, RETRIES, url, e)
        except Exception as e:
            logger.warning("Unexpected error (attempt %d/%d) -> %s: %s", attempt, RETRIES, url, e)

        if attempt < RETRIES:
            await asyncio.sleep(RETRY_DELAY * (2 ** (attempt - 1)))  # exponential back-off

    return None


# ---------- WORKER ----------
async def process_product(
    session: aiohttp.ClientSession,
    semaphore: asyncio.Semaphore,
    product: Product,
    output_dir: Path,
    stats: Stats,
    logger: logging.Logger,
) -> None:
    filename  = build_filename(product)
    file_path = output_dir / filename

    if SKIP_EXISTING and file_path.exists():
        logger.info("Skip (exists): %s", filename)
        stats.skipped += 1
        return

    async with semaphore:
        html = await fetch_html(session, product.product_url, logger)

    if html is None:
        logger.error("Failed: %s -> %s", product.product_name, product.product_url)
        stats.failed += 1
        return

    file_path.write_text(html, encoding="utf-8")
    logger.info("Saved : %s", filename)
    stats.success += 1


# ---------- MAIN ----------
async def main() -> None:
    logger = setup_logging()

    products = load_products(INPUT_FILE)
    if not products:
        logger.warning("No valid products found in %s", INPUT_FILE)
        return

    output_dir = Path(OUTPUT_DIR)
    output_dir.mkdir(parents=True, exist_ok=True)

    stats     = Stats(total=len(products))
    semaphore = asyncio.Semaphore(CONCURRENCY)
    timeout   = aiohttp.ClientTimeout(total=TIMEOUT)
    connector = aiohttp.TCPConnector(limit=CONCURRENCY, ssl=False)

    logger.info("Scraping %d products -> %s", stats.total, output_dir)

    async with aiohttp.ClientSession(connector=connector, timeout=timeout, headers=HEADERS) as session:
        tasks = [
            process_product(session, semaphore, product, output_dir, stats, logger)
            for product in products
        ]
        await asyncio.gather(*tasks)

    print(stats.summary())


# ---------- RUN (Colab-safe) ----------
await main()

import shutil
import os

# Create a zip archive
shutil.make_archive(output_zip_file, 'zip', folder_to_zip)

print(f'Folder "{folder_to_zip}" successfully zipped to "{output_zip_file}.zip"')

Streaming output truncated to the last 5000 lines.
11:39:34 [INFO] Saved : fiche_male_110721.html
11:39:35 [INFO] Saved : capot_bleu_022372.html
11:39:35 [INFO] Saved : capot_gris_022371.html
11:39:35 [INFO] Saved : capot_electrique_blanc_022389.html
11:39:35 [INFO] Saved : cable_fiche_110773.html
11:39:35 [INFO] Saved : conn_vanne_sunag07_8_110769.html
11:39:35 [INFO] Saved : capot_gris_022374.html
11:39:35 [INFO] Saved : etrier_de_fixation_x1_022379.html
11:39:35 [INFO] Saved : connecteur_12_plots_110770.html
11:39:35 [INFO] Saved : connecteur_110774.html
11:39:35 [INFO] Saved : flexibles_2x1m_raccords_022392.html
11:39:35 [INFO] Saved : patte_d_accrochage_022397.html
11:39:35 [INFO] Saved : ventilateur_cv5_vs_filerie_026524.html
11:39:35 [INFO] Saved : commande_digitale_026529.html
11:39:35 [INFO] Saved : ventilateur_ac_220_026527.html
11:39:35 [INFO] Saved : faisceau_commande_digitale_ihm_026526.html
11:39:35 [INFO] Saved : rail_l100mm_022393.html
11:39:35 [INFO] Saved : rondelle_m


─────────────────────────────────────────────
  Total    : 21012
  ✅ Saved  : 21012
  ⏭  Skipped: 0
  ❌ Failed : 0
  Time     : 1309.7s
─────────────────────────────────────────────
Folder "/content/fujitsu_products_html" successfully zipped to "/content/fujitsu_products_html.zip"


In [11]:
import shutil
import os

source_file = "/content/fujitsu_products_html.zip"
destination_folder = "/content/drive/MyDrive/Upwork/kevin"

# Ensure the destination folder exists
os.makedirs(destination_folder, exist_ok=True)

# Move the zip file
shutil.move(source_file, destination_folder)

print(f"Successfully moved '{source_file}' to '{destination_folder}'")

Successfully moved '/content/fujitsu_products_html.zip' to '/content/drive/MyDrive/Upwork/kevin'


In [12]:
import os
import json
import csv
from bs4 import BeautifulSoup

INPUT_FOLDER = "nexa_pages"
BASE_URL = "https://piece-climatisation.com"  # <-- change this

OUTPUT_JSON = "nexa_products.json"
OUTPUT_CSV = "nexa_products.csv"


def clean_price(price_text):
    if not price_text:
        return ""
    return (
        price_text.replace("\xa0", " ")
        .replace("€", "")
        .replace("TTC", "")
        .strip()
    )


def parse_html_file(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f, "html.parser")

    products = []

    items = soup.select(".product-list-element")

    for item in items:
        try:
            # product name
            name_tag = item.select_one(".product-name b")
            name = name_tag.get_text(strip=True) if name_tag else ""

            # product URL
            link_tag = item.select_one(".product-name")
            relative_url = link_tag["href"] if link_tag else ""
            product_url = BASE_URL + relative_url if relative_url else ""

            # reference + brand
            ref_brand_text = item.select_one(".description p")
            ref = ""
            brand = ""

            if ref_brand_text:
                lines = ref_brand_text.get_text("\n").split("\n")
                for line in lines:
                    line = line.strip()
                    if line.startswith("Ref:"):
                        ref = line.replace("Ref:", "").strip()
                    elif line:
                        brand = line.strip()

            # price
            price_tag = item.select_one(".final-price")
            price = clean_price(price_tag.get_text()) if price_tag else ""

            products.append({
                "product_name": name,
                "product_url": product_url,
                "brand": brand,
                "reference": ref,
                "price": price
            })

        except Exception as e:
            print(f"Error parsing item in {filepath}: {e}")

    return products


def main():
    all_products = []

    files = [f for f in os.listdir(INPUT_FOLDER) if f.endswith(".html")]
    files.sort()  # ensure consistent order

    print(f"📂 Total files: {len(files)}")

    for idx, filename in enumerate(files, 1):
        filepath = os.path.join(INPUT_FOLDER, filename)
        print(f"Processing {idx}/{len(files)}: {filename}")

        products = parse_html_file(filepath)
        all_products.extend(products)

    print(f"\n✅ Total products scraped: {len(all_products)}")

    # Save JSON
    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(all_products, f, indent=2, ensure_ascii=False)

    # Save CSV
    with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=[
            "product_name",
            "product_url",
            "brand",
            "reference",
            "price"
        ])
        writer.writeheader()
        writer.writerows(all_products)

    print(f"\n💾 Saved to {OUTPUT_JSON} and {OUTPUT_CSV}")


if __name__ == "__main__":
    main()

📂 Total files: 3481
Processing 1/3481: boutique-piece-detachee-climatisation_fournisseur_nexa.html
Processing 2/3481: boutique-piece-detachee-climatisation_fournisseur_nexa_page_10.html
Processing 3/3481: boutique-piece-detachee-climatisation_fournisseur_nexa_page_100.html
Processing 4/3481: boutique-piece-detachee-climatisation_fournisseur_nexa_page_1000.html
Processing 5/3481: boutique-piece-detachee-climatisation_fournisseur_nexa_page_1001.html
Processing 6/3481: boutique-piece-detachee-climatisation_fournisseur_nexa_page_1002.html
Processing 7/3481: boutique-piece-detachee-climatisation_fournisseur_nexa_page_1003.html
Processing 8/3481: boutique-piece-detachee-climatisation_fournisseur_nexa_page_1004.html
Processing 9/3481: boutique-piece-detachee-climatisation_fournisseur_nexa_page_1005.html
Processing 10/3481: boutique-piece-detachee-climatisation_fournisseur_nexa_page_1006.html
Processing 11/3481: boutique-piece-detachee-climatisation_fournisseur_nexa_page_1007.html
Processing 1

In [13]:
import asyncio
import aiohttp
import json
import logging
import re
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional


# ================================================================
#  CONFIG  ← edit these values before running
# ================================================================
INPUT_FILE    = "nexa_products.json"  # path to your JSON file
OUTPUT_DIR    = "nexa_products_html"            # folder to save .html files
CONCURRENCY   = 10                      # max parallel requests
TIMEOUT       = 30                      # seconds per request
RETRIES       = 3                       # attempts per URL
RETRY_DELAY   = 1.5                     # exponential back-off base (seconds)
SKIP_EXISTING = True                    # skip URLs already saved to disk
VERBOSE       = False                   # set True for debug logs
# ================================================================

folder_to_zip = '/content/nexa_products_html'
output_zip_file = '/content/nexa_products_html'

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
}


# ---------- LOGGING ----------
def setup_logging() -> logging.Logger:
    level = logging.DEBUG if VERBOSE else logging.INFO
    logging.basicConfig(
        level=level,
        format="%(asctime)s [%(levelname)s] %(message)s",
        datefmt="%H:%M:%S",
        force=True,  # re-apply in Colab which pre-configures logging
    )
    return logging.getLogger("scraper")


# ---------- DATA MODELS ----------
@dataclass
class Product:
    product_name: str
    product_url: str
    reference: str = ""
    brand: str = ""
    price: str = ""
    availability: str = ""

    @classmethod
    def from_dict(cls, data: dict) -> "Product":
        required = {"product_name", "product_url"}
        missing = required - data.keys()
        if missing:
            raise ValueError(f"Missing required fields: {missing}")
        return cls(**{k: data.get(k, "") for k in cls.__dataclass_fields__})


@dataclass
class Stats:
    total: int = 0
    success: int = 0
    failed: int = 0
    skipped: int = 0
    start_time: float = field(default_factory=time.time)

    def elapsed(self) -> float:
        return time.time() - self.start_time

    def summary(self) -> str:
        return (
            f"\n{'─'*45}\n"
            f"  Total    : {self.total}\n"
            f"  ✅ Saved  : {self.success}\n"
            f"  ⏭  Skipped: {self.skipped}\n"
            f"  ❌ Failed : {self.failed}\n"
            f"  Time     : {self.elapsed():.1f}s\n"
            f"{'─'*45}"
        )


# ---------- HELPERS ----------
def safe_filename(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    return text.strip("_")[:80]


def build_filename(product: Product) -> str:
    parts = [safe_filename(product.product_name)]
    if product.reference:
        parts.append(safe_filename(product.reference))
    return "_".join(parts) + ".html"


def load_products(json_path: str) -> list[Product]:
    path = Path(json_path)
    if not path.exists():
        raise FileNotFoundError(f"Input file not found: {json_path}")
    if path.suffix.lower() != ".json":
        raise ValueError(f"Expected a .json file, got: {json_path}")

    with open(path, encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, list):
        raise ValueError("JSON root must be a list of product objects.")

    products, errors = [], []
    for i, item in enumerate(data):
        try:
            products.append(Product.from_dict(item))
        except (ValueError, TypeError) as e:
            errors.append(f"  Item #{i}: {e}")

    if errors:
        logging.getLogger("scraper").warning(
            "Skipped %d invalid item(s):\n%s", len(errors), "\n".join(errors)
        )
    return products


# ---------- FETCH ----------
async def fetch_html(
    session: aiohttp.ClientSession,
    url: str,
    logger: logging.Logger,
) -> Optional[str]:
    for attempt in range(1, RETRIES + 1):
        try:
            async with session.get(url) as response:
                if response.status == 200:
                    return await response.text()
                if response.status in (403, 404, 410):
                    logger.warning("HTTP %s (no retry) -> %s", response.status, url)
                    return None
                logger.warning("HTTP %s (attempt %d/%d) -> %s", response.status, attempt, RETRIES, url)
        except asyncio.TimeoutError:
            logger.warning("Timeout (attempt %d/%d) -> %s", attempt, RETRIES, url)
        except aiohttp.ClientConnectorError as e:
            logger.warning("Connection error (attempt %d/%d) -> %s: %s", attempt, RETRIES, url, e)
        except Exception as e:
            logger.warning("Unexpected error (attempt %d/%d) -> %s: %s", attempt, RETRIES, url, e)

        if attempt < RETRIES:
            await asyncio.sleep(RETRY_DELAY * (2 ** (attempt - 1)))  # exponential back-off

    return None


# ---------- WORKER ----------
async def process_product(
    session: aiohttp.ClientSession,
    semaphore: asyncio.Semaphore,
    product: Product,
    output_dir: Path,
    stats: Stats,
    logger: logging.Logger,
) -> None:
    filename  = build_filename(product)
    file_path = output_dir / filename

    if SKIP_EXISTING and file_path.exists():
        logger.info("Skip (exists): %s", filename)
        stats.skipped += 1
        return

    async with semaphore:
        html = await fetch_html(session, product.product_url, logger)

    if html is None:
        logger.error("Failed: %s -> %s", product.product_name, product.product_url)
        stats.failed += 1
        return

    file_path.write_text(html, encoding="utf-8")
    logger.info("Saved : %s", filename)
    stats.success += 1


# ---------- MAIN ----------
async def main() -> None:
    logger = setup_logging()

    products = load_products(INPUT_FILE)
    if not products:
        logger.warning("No valid products found in %s", INPUT_FILE)
        return

    output_dir = Path(OUTPUT_DIR)
    output_dir.mkdir(parents=True, exist_ok=True)

    stats     = Stats(total=len(products))
    semaphore = asyncio.Semaphore(CONCURRENCY)
    timeout   = aiohttp.ClientTimeout(total=TIMEOUT)
    connector = aiohttp.TCPConnector(limit=CONCURRENCY, ssl=False)

    logger.info("Scraping %d products -> %s", stats.total, output_dir)

    async with aiohttp.ClientSession(connector=connector, timeout=timeout, headers=HEADERS) as session:
        tasks = [
            process_product(session, semaphore, product, output_dir, stats, logger)
            for product in products
        ]
        await asyncio.gather(*tasks)

    print(stats.summary())


# ---------- RUN (Colab-safe) ----------
await main()

import shutil
import os

# Create a zip archive
shutil.make_archive(output_zip_file, 'zip', folder_to_zip)

print(f'Folder "{folder_to_zip}" successfully zipped to "{output_zip_file}.zip"')

Streaming output truncated to the last 5000 lines.
12:13:36 [INFO] Saved : join_pas_es_12_mult_2_00psg000020100a.html
12:13:36 [INFO] Saved : joint_es_14_mult_2_00psg000020500a.html
12:13:36 [INFO] Saved : joint_echangeur_mult_2_00psg000020200a.html
12:13:36 [INFO] Saved : plaque_de_tete_avec_raccords_soudes_00psg000019900ercd.html
12:13:37 [INFO] Saved : joint_echangeur_mult_2_00psg000020600a.html
12:13:37 [INFO] Saved : capacitor_p_291_0504_s.html
12:13:37 [INFO] Saved : condensateur_p_291_1004_s.html
12:13:37 [INFO] Saved : condensateur_440v_10_10_mfd_p_291_1014_s.html
12:13:37 [INFO] Saved : run_cap_oval_440v_7_5mfd_p_291_0774_s.html
12:13:37 [INFO] Saved : capacitor_p_291_0754_s.html
12:13:37 [INFO] Saved : condensateur_p_291_1504_s.html
12:13:37 [INFO] Saved : run_cap_oval_440v_20_5mfd_p_291_2054_s.html
12:13:37 [INFO] Saved : condensateur_p_291_3074_s.html
12:13:37 [INFO] Saved : condensateur_p_291_2504_s.html
12:13:37 [INFO] Saved : epingle_ax_11ar_168_s.html
12:13:37 [INFO] Sa


─────────────────────────────────────────────
  Total    : 31329
  ✅ Saved  : 31329
  ⏭  Skipped: 0
  ❌ Failed : 0
  Time     : 1916.1s
─────────────────────────────────────────────
Folder "/content/nexa_products_html" successfully zipped to "/content/nexa_products_html.zip"


In [14]:
import shutil
import os

source_file = "/content/nexa_products_html.zip"
destination_folder = "/content/drive/MyDrive/Upwork/kevin"

# Ensure the destination folder exists
os.makedirs(destination_folder, exist_ok=True)

# Move the zip file
shutil.move(source_file, destination_folder)

print(f"Successfully moved '{source_file}' to '{destination_folder}'")

Successfully moved '/content/nexa_products_html.zip' to '/content/drive/MyDrive/Upwork/kevin'
